# Bab 7: Membangun Aplikasi Sembang
## Permulaan Pantas API OpenAI

Buku nota ini diadaptasi daripada [Repositori Sampel Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang termasuk buku nota yang mengakses perkhidmatan [Azure OpenAI](notebook-azure-openai.ipynb).

API OpenAI Python juga berfungsi dengan Model Azure OpenAI, dengan beberapa pengubahsuaian. Ketahui lebih lanjut mengenai perbezaan di sini: [Cara bertukar antara OpenAI dan titik akhir Azure OpenAI dengan Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Gambaran Keseluruhan  
"Model bahasa besar adalah fungsi yang memetakan teks ke teks. Diberi satu rentetan teks input, model bahasa besar cuba meramalkan teks yang akan datang seterusnya"(1). Buku nota "panduan cepat" ini akan memperkenalkan pengguna kepada konsep LLM peringkat tinggi, keperluan pakej teras untuk memulakan dengan AML, pengenalan ringkas kepada reka bentuk arahan, dan beberapa contoh ringkas bagi pelbagai kes penggunaan. 


## Jadual Kandungan  

[Gambaran Keseluruhan](#overview)  
[Cara menggunakan Perkhidmatan OpenAI](#how-to-use-openai-service)  
[1. Mencipta Perkhidmatan OpenAI anda](#1.-creating-your-openai-service)  
[2. Pemasangan](#2.-installation)    
[3. Kelayakan](#3.-credentials)  

[Kes Penggunaan](#use-cases)    
[1. Meringkaskan Teks](#1.-summarize-text)  
[2. Mengklasifikasikan Teks](#2.-classify-text)  
[3. Menjana Nama Produk Baru](#3.-generate-new-product-names)  
[4. Melaras Penilai](#4.fine-tune-a-classifier)  

[Rujukan](#references)


### Bina arahan pertama anda  
Latihan ringkas ini akan memberikan pengenalan asas untuk menghantar arahan kepada model OpenAI untuk tugasan mudah "meringkaskan".


**Langkah-langkah**:  
1. Pasang perpustakaan OpenAI dalam persekitaran python anda  
2. Muatkan perpustakaan pembantu standard dan tetapkan kelayakan keselamatan OpenAI biasa anda untuk Perkhidmatan OpenAI yang telah anda cipta  
3. Pilih model untuk tugasan anda  
4. Buat arahan ringkas untuk model  
5. Hantar permintaan anda ke API model!


### 1. Pasang OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Import perpustakaan pembantu dan cipta kelayakan


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Mencari model yang sesuai  
Model seperti GPT-4o dan GPT-4o mini boleh memahami dan menjana bahasa semula jadi, dan tersedia di platform OpenAI dengan pelbagai tahap kuasa dan kelajuan yang sesuai untuk tugasan yang berbeza.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Reka Bentuk Prompt  

"Keajaiban model bahasa besar adalah dengan dilatih untuk meminimumkan ralat ramalan ini ke atas jumlah teks yang banyak, model akan belajar konsep yang berguna untuk ramalan ini. Sebagai contoh, mereka mempelajari konsep seperti"(1):

* cara mengeja
* bagaimana tatabahasa berfungsi
* bagaimana memparafrasekan
* bagaimana menjawab soalan
* bagaimana mengadakan perbualan
* bagaimana menulis dalam banyak bahasa
* bagaimana menulis kod
* dan sebagainya.

#### Cara mengawal model bahasa besar  
"Daripada semua input kepada model bahasa besar, input yang paling berpengaruh ialah prompt teks(1).

Model bahasa besar boleh diberi arahan untuk menghasilkan output dengan beberapa cara:

Arahan: Beritahu model apa yang anda mahu
Lengkapkan: Galakkan model untuk melengkapkan permulaan apa yang anda mahu
Demonstrasi: Tunjukkan model apa yang anda mahu, dengan sama ada:
Beberapa contoh dalam prompt
Beratus-ratus atau ribuan contoh dalam set data latihan penalaan halus"



#### Terdapat tiga garis panduan asas untuk membuat prompt:

**Tunjuk dan beritahu**. Jelaskan apa yang anda mahu sama ada melalui arahan, contoh, atau gabungan kedua-duanya. Jika anda mahu model mengaturkan senarai item mengikut susunan abjad atau mengklasifikasikan perenggan mengikut sentimen, tunjukkan itu yang anda mahu.

**Sediakan data berkualiti**. Jika anda cuba membina pengklasifikasi atau mahu model mengikuti corak, pastikan terdapat cukup contoh. Pastikan anda menyemak contoh anda — model biasanya cukup pintar untuk melihat kesilapan ejaan asas dan memberi anda jawapan, tetapi ia juga mungkin menganggap ini sengaja dan ia boleh menjejaskan jawapan.

**Semak tetapan anda.** Tetapan suhu dan top_p mengawal sejauh mana deterministik model dalam menghasilkan jawapan. Jika anda meminta jawapan di mana hanya satu jawapan betul, maka anda mahu menetapkan ini lebih rendah. Jika anda mahukan jawapan yang lebih pelbagai, anda mungkin mahu menetapkannya lebih tinggi. Kesilapan nombor satu yang dibuat orang dengan tetapan ini ialah menganggap ia adalah kawalan “kepintaran” atau “kreativiti”.


Sumber: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Hantar!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Rumuskan Teks  
#### Cabaran  
Rumuskan teks dengan menambah 'tl;dr:' di akhir petikan teks. Perhatikan bagaimana model memahami cara melakukan pelbagai tugas tanpa arahan tambahan. Anda boleh mencuba dengan arahan yang lebih deskriptif daripada tl;dr untuk mengubah tingkah laku model dan menyesuaikan rumusan yang anda terima(3).  

Kerja terkini telah menunjukkan peningkatan ketara pada banyak tugas dan penanda aras NLP dengan pra-latihan pada korpus teks yang besar diikuti dengan penyetelan halus pada tugas tertentu. Walaupun biasanya bersifat tugas-agnostik dalam seni bina, kaedah ini masih memerlukan set data penyetelan halus khusus tugas yang terdiri daripada ribuan atau puluhan ribu contoh. Sebaliknya, manusia secara amnya boleh melaksanakan tugas bahasa baru hanya dengan beberapa contoh atau arahan mudah - sesuatu yang sistem NLP semasa masih banyak menghadapi kesukaran untuk lakukan. Di sini kami menunjukkan bahawa peningkatan saiz model bahasa dapat memperbaiki prestasi tugas-agnostik, beberapa tembakan, kadangkala mencapai tahap daya saing dengan pendekatan penyetelan halus terkemuka sebelum ini. 



Tl;dr


# Latihan untuk beberapa kes penggunaan  
1. Rumuskan Teks  
2. Klasifikasikan Teks  
3. Jana Nama Produk Baru


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasifikasikan Teks  
#### Cabaran  
Klasifikasikan item ke dalam kategori yang diberikan semasa masa inferens. Dalam contoh berikut, kami menyediakan kedua-dua kategori dan teks untuk diklasifikasikan dalam prompt(*playground_reference). 

Pertanyaan Pelanggan: Hai, salah satu kekunci pada papan kekunci komputer riba saya baru-baru ini rosak dan saya memerlukan pengganti:

Kategori klasifikasi:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Hasilkan Nama Produk Baru
#### Cabaran
Cipta nama produk dari perkataan contoh. Di sini kami sertakan dalam arahan maklumat tentang produk yang akan kami hasilkan nama untuknya. Kami juga menyediakan contoh serupa untuk menunjukkan corak yang ingin kami terima. Kami juga menetapkan nilai suhu tinggi untuk meningkatkan kerandoman dan respons yang lebih inovatif.

Penerangan produk: Pembuat milkshake rumah
Perkataan benih: pantas, sihat, padat.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Penerangan produk: Sepasang kasut yang boleh muat saiz kaki apa pun.
Perkataan benih: boleh disesuaikan, sesuai, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Rujukan  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Amalan terbaik untuk penalaan halus model GPT bagi mengklasifikasi teks](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Untuk Bantuan Lanjut  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Penyumbang
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
